# 🚀 Paper Tutor — GPU Inference Server

This notebook launches a **FastAPI inference server** on Colab's free T4 GPU.
Your local Streamlit app sends prompts here for fast GPU-accelerated generation.

**Architecture:**
```
Local Machine          →  ngrok tunnel  →  Colab T4 GPU
(Streamlit + FAISS)    →  HTTP POST     →  (Qwen2.5 4-bit)
```

**Setup time:** ~3-4 minutes (one-time model download)

---

## ⚠️ Prerequisites
1. **Runtime → Change runtime type → T4 GPU**
2. Get a free ngrok auth token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)

## Step 1: Install Dependencies

In [ ]:
%%time
# Install all required packages
!pip install -q \
    torch \
    transformers>=4.40.0 \
    accelerate>=0.30.0 \
    bitsandbytes>=0.43.0 \
    fastapi>=0.110.0 \
    uvicorn>=0.29.0 \
    pyngrok>=7.1.0 \
    pydantic>=2.0.0

print('\n✅ Dependencies installed!')

## Step 2: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu_name}')
    print(f'✅ VRAM: {vram_gb:.1f} GB')
else:
    print('❌ No GPU detected!')
    print('   Go to Runtime → Change runtime type → T4 GPU')
    raise RuntimeError('GPU required for inference server')

## Step 3: Configure ngrok

Get your free auth token from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ⚠️ PASTE YOUR NGROK AUTH TOKEN HERE
NGROK_AUTH_TOKEN = ''  # <-- Paste your token between the quotes

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        'Please set your ngrok auth token above!\n'
        'Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken'
    )

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('✅ ngrok configured!')

## Step 4: Write the FastAPI Server

This creates the server file directly in Colab.

In [ ]:
%%writefile colab_server.py
"""
FastAPI GPU Inference Server — Runs on Google Colab T4
"""

import os
import time
import json
import logging
from contextlib import asynccontextmanager

import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
from threading import Thread

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Config ---
MODEL_ID = os.getenv('MODEL_ID', 'Qwen/Qwen2.5-1.5B-Instruct')
MAX_MODEL_LEN = 2048

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

# --- Globals ---
model = None
tokenizer = None
device = None


def load_model():
    global model, tokenizer, device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    logger.info(f'Loading {MODEL_ID} on {device}')

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {'trust_remote_code': True, 'torch_dtype': torch.float16}
    if device == 'cuda':
        load_kwargs['quantization_config'] = QUANTIZATION_CONFIG
        load_kwargs['device_map'] = 'auto'

    start = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
    model.eval()
    t = time.perf_counter() - start

    if device == 'cuda':
        vram = torch.cuda.memory_allocated() / 1e9
        logger.info(f'Model loaded in {t:.1f}s — VRAM: {vram:.2f} GB')
    else:
        logger.info(f'Model loaded in {t:.1f}s (CPU)')


@asynccontextmanager
async def lifespan(app):
    load_model()
    yield


app = FastAPI(title='Paper Tutor GPU Server', version='1.0.0', lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*'],
)


class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = Field(default=512, ge=1, le=2048)
    temperature: float = Field(default=0.3, ge=0.0, le=2.0)
    top_p: float = Field(default=0.9, ge=0.0, le=1.0)
    repetition_penalty: float = Field(default=1.1, ge=1.0, le=2.0)
    stream: bool = False


@app.get('/health')
async def health():
    info = {'status': 'healthy', 'model': MODEL_ID, 'device': device}
    if device == 'cuda':
        info['vram_allocated_gb'] = round(torch.cuda.memory_allocated() / 1e9, 2)
        info['vram_total_gb'] = round(torch.cuda.get_device_properties(0).total_mem / 1e9, 2)
    return info


@app.post('/generate')
async def generate(req: GenerateRequest):
    if model is None:
        raise HTTPException(503, 'Model not loaded')
    if req.stream:
        return StreamingResponse(
            _stream_gen(req), media_type='text/event-stream',
            headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'},
        )
    return _batch_gen(req)


def _batch_gen(req):
    start = time.perf_counter()
    inputs = tokenizer(
        req.prompt, return_tensors='pt', truncation=True,
        max_length=MAX_MODEL_LEN - req.max_tokens,
    )
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    input_len = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=req.max_tokens,
            temperature=max(req.temperature, 0.01),
            top_p=req.top_p, repetition_penalty=req.repetition_penalty,
            do_sample=req.temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    gen_ms = (time.perf_counter() - start) * 1000
    tps = len(new_tokens) / (gen_ms / 1000) if gen_ms > 0 else 0
    logger.info(f'{len(new_tokens)} tokens in {gen_ms:.0f}ms ({tps:.0f} tok/s)')
    return {'response': text.strip(), 'tokens_generated': len(new_tokens),
            'generation_time_ms': round(gen_ms, 1), 'model': MODEL_ID}


async def _stream_gen(req):
    inputs = tokenizer(
        req.prompt, return_tensors='pt', truncation=True,
        max_length=MAX_MODEL_LEN - req.max_tokens,
    )
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = {
        **inputs, 'max_new_tokens': req.max_tokens,
        'temperature': max(req.temperature, 0.01),
        'top_p': req.top_p, 'repetition_penalty': req.repetition_penalty,
        'do_sample': req.temperature > 0,
        'pad_token_id': tokenizer.pad_token_id, 'streamer': streamer,
    }
    thread = Thread(target=lambda: _gen_thread(kwargs))
    thread.start()
    for tok in streamer:
        if tok:
            yield f'data: {json.dumps({"token": tok})}\n\n'
    yield 'data: [DONE]\n\n'
    thread.join()


def _gen_thread(kwargs):
    with torch.inference_mode():
        model.generate(**kwargs)

## Step 5: Launch the Server + ngrok Tunnel

This cell starts the FastAPI server and creates a public ngrok tunnel.

**Copy the printed URL** and paste it into your Paper Tutor Streamlit sidebar.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Kill any existing server/tunnel
ngrok.kill()

# Start the FastAPI server in background
server_process = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'colab_server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to start
print('⏳ Loading model (first time may take 2-3 minutes)...')
time.sleep(30)  # Model loading takes ~20-30s on T4

# Verify server is running
import requests
for i in range(10):
    try:
        r = requests.get('http://localhost:8000/health', timeout=5)
        if r.status_code == 200:
            health = r.json()
            print(f'\n✅ Server healthy!')
            print(f'   Model: {health["model"]}')
            print(f'   Device: {health["device"]}')
            if 'vram_allocated_gb' in health:
                print(f'   VRAM: {health["vram_allocated_gb"]} / {health["vram_total_gb"]} GB')
            break
    except:
        time.sleep(10)
else:
    print('❌ Server failed to start. Check logs:')
    print(server_process.stderr.read().decode())
    raise RuntimeError('Server startup failed')

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f'\n{"="*60}')
print(f'🌐 YOUR INFERENCE URL:')
print(f'')
print(f'   {public_url}')
print(f'')
print(f'   Copy this URL into the Paper Tutor Streamlit sidebar!')
print(f'{"="*60}')

## Step 6: Test the Server

Quick test to verify generation works:

In [ ]:
import requests, time

# Test non-streaming generation
test_prompt = (
    'You are an ML research tutor.\n\n'
    'Context: The Transformer uses self-attention to process sequences in parallel.\n\n'
    'Question: What is self-attention?'
)

print('📝 Testing non-streaming generation...')
start = time.time()
resp = requests.post(
    'http://localhost:8000/generate',
    json={'prompt': test_prompt, 'max_tokens': 128, 'stream': False},
    timeout=60,
)
elapsed = time.time() - start
data = resp.json()

print(f'\n🤖 Response ({elapsed:.1f}s):')
print(data['response'][:500])
print(f'\n📊 Stats: {data["tokens_generated"]} tokens, {data["generation_time_ms"]:.0f}ms')

In [ ]:
# Test streaming generation
print('📝 Testing streaming generation...\n')
print('🤖 Response: ', end='', flush=True)

resp = requests.post(
    'http://localhost:8000/generate',
    json={'prompt': test_prompt, 'max_tokens': 128, 'stream': True},
    stream=True, timeout=60,
)

import json
for line in resp.iter_lines():
    if line:
        text = line.decode('utf-8')
        if text.startswith('data: ') and text[6:].strip() != '[DONE]':
            try:
                token = json.loads(text[6:])['token']
                print(token, end='', flush=True)
            except:
                pass
print('\n\n✅ Streaming works!')

## Step 7: Monitor (Optional)

Run this cell periodically to check GPU memory usage:

In [ ]:
!nvidia-smi

---

## 📋 Quick Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Server status, model info, VRAM usage |
| `/generate` | POST | Text generation (streaming or batch) |

### Local Setup (on your machine)

1. In Streamlit sidebar → toggle **⚡ GPU Inference** ON
2. Paste the ngrok URL into the text input
3. Ask questions as normal — inference now runs on this Colab GPU!

### Keep This Notebook Running
- Don't close this tab while using the Paper Tutor
- Colab free tier sessions last ~90 minutes
- If it disconnects, just re-run all cells